# Linear Regression

**Lab 3 · Linear regression with PyTorch**

This lab builds on the Python and PyTorch notebooks from Weeks 1 and 2. Work in **Google Colab**; no GPU is needed. This notebook creates its own data and does not need variables from the earlier notebooks.

Work from top to bottom: each question uses results from the previous ones. Try each question in its code cell before opening **Show answer**. To use a supplied answer, copy its code into the corresponding code cell and run it; opening an answer box does not execute its code.

If you get stuck, use the supplied answer for that question and continue. The later sections are written so that you can still complete the lab after copying in a reference solution.

We begin with a short plotting reminder, then create the dataset used throughout the regression exercises.


## Plotting reminder: the same commands with PyTorch tensors

In Lab 1 we passed Python lists to Matplotlib. Here we use ordinary **CPU PyTorch tensors without automatic differentiation**. For these tensors the plotting commands are unchanged: no NumPy code or conversion is needed.

`plt.plot(x_values, y_values)` joins the supplied coordinate pairs; `plt.scatter(x_values, y_values)` draws dots. Both inputs must have matching lengths. Put both calls before one `plt.show()` to overlay them, and use `xlabel` and `ylabel` to name the axes.

**Run the example below; it is supplied, not another exercise.** `t.linspace(-2, 2, 7)` creates seven equally spaced inputs, and `2*x_demo + 1` computes a value for each input. Expect seven dots on an increasing straight line. These demonstration values are separate from the lab dataset.


In [ ]:
import torch as t
import matplotlib.pyplot as plt

x_demo = t.linspace(-2, 2, 7)
y_demo = 2 * x_demo + 1

plt.plot(x_demo, y_demo)       # A line joining the tensor's coordinate pairs.
plt.scatter(x_demo, y_demo)    # The seven individual points on that same graph.
plt.xlabel("x")
plt.ylabel("y")
plt.show()


### Now create the regression dataset

In the exercises we will use `plt.scatter(x, y)` for the **observed data** and `plt.plot(x, yh)` for the **model predictions**. Here `yh` will be the vector of predictions that you compute; it is not a special Matplotlib command. These tensors will all have shape `(N,)`.

The next cell defines the actual inputs `x` and targets `y` and draws their scatter plot. Later plotting cells use these names, **not** `x_demo` and `y_demo`. The input values are already in increasing order, so joining model predictions will draw the fitted curve from left to right. Drawing a curve does not fit it: the fitting calculations come next.


In [ ]:
import torch as t
import matplotlib.pyplot as plt

t.manual_seed(1)  # Repeat the random draws when rerunning this cell in the same setup.

N = 100
x = t.linspace(-2, 2, N)  # N equally spaced inputs, including -2 and 2; shape (N,).
# Each entry of t.randn(N) is a standard-normal random draw.
# The last term adds noise to the quadratic curve 3 + x + 0.2*x**2.
y = 3. + x + 0.2*x**2 + 0.3*t.randn(N)

plt.scatter(x, y)  # One point for each observed pair (x_i, y_i).
plt.xlabel('x')
plt.ylabel('y')
plt.show()


## 1) Fitting a straight line

### 1.1) Constructing the design matrix

We start by fitting the affine predictor

$$
f^{\mathrm{aff}}_{\boldsymbol{\theta}}(x)=b+ax,
\qquad
\boldsymbol{\theta}
=
\begin{pmatrix}
b\\
a
\end{pmatrix}.
$$

The first step is to construct the **design matrix**. Each row represents one observation. The first column contains the constant feature $1$, which multiplies the intercept $b$, and the second column contains the input values $x_i$, which multiply the slope $a$:

$$
\mathbf{X}=
\begin{pmatrix}
1 & x_1\\
1 & x_2\\
\vdots & \vdots\\
1 & x_N
\end{pmatrix}.
$$

Use `X` for this matrix. Lowercase `x` remains the vector of input values.

In the lecture, $\boldsymbol{\theta}$ is written as a $2\times 1$ column. In PyTorch, we will store the same two coefficients in a 1-dimensional tensor of shape `(2,)`. With `X` of shape `(N, 2)`, the product `X @ theta` then has shape `(N,)`, giving one prediction for each observation.

**Hint:** use [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html). Check that `X.shape` is `(N, 2)`: the features should be columns, not rows.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.1</summary>

```python
ones = t.ones(N)
# Both inputs have shape (N,). Stack them along a NEW last dimension.
X = t.stack([ones, x], dim=-1)  # Shape (N, 2); each row is [1, x_i].
print(X.shape)
```

The output is `torch.Size([100, 2])`. Here `dim=-1` places the new dimension last, so the two input vectors become the two columns of `X`.

</details>


### 1.2) Finding the best parameters

Find the parameter vector that minimises the sum of squared residuals using the analytic formula

$$
\boldsymbol{\theta}^*
=
(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}.
$$

This formula requires linearly independent columns of $\mathbf{X}$, as we have here. For our two-dimensional tensor, `X.mT` is the transpose of `X`. It is accessed **without parentheses**.

Think about how to group the matrix products efficiently, using the shapes of the intermediate results. Store the parameters in `theta` and print them. With `y` of shape `(N,)`, `theta` should have shape `(2,)`. Its first entry is the fitted intercept and its second entry is the fitted slope.

Expect a positive slope around $1$. The intercept need not equal the generating value $3$: we are fitting a straight line, whereas the data also contain a quadratic term and noise.

**Hint:** find a matrix-inverse function in the [PyTorch documentation](https://docs.pytorch.org/docs/stable/index.html). Looking up an unfamiliar operation is part of this exercise. You do not need to read the whole documentation.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.2</summary>

```python
# t.linalg.inv(...) computes a matrix inverse.
# X.mT @ X has shape (2, 2), while X.mT @ y has shape (2,).
theta = t.linalg.inv(X.mT @ X) @ (X.mT @ y)
print(theta)
```

Computing `X.mT @ y` first avoids forming the additional `(2, N)` intermediate that would result from first multiplying the inverse by `X.mT`.

</details>


### 1.3) Plotting predictions using the explicit formula

Plot the fitted line together with the original data, first using

$$
f^{\mathrm{aff}}_{\boldsymbol{\theta}}(x)=b+ax.
$$

Since `theta[0]` stores $b$ and `theta[1]` stores $a$, the prediction at the observed inputs is computed as `theta[0] + theta[1] * x`.

Use `yh` as a short code name for the predicted values.

**Plotting reminder:** after computing `yh`, use `plt.plot(x, yh)` and `plt.scatter(x, y)` in the same cell, add `plt.xlabel("x")` and `plt.ylabel("y")`, and finish with `plt.show()`. The line is the fitted prediction and the dots are the observations. Both `x` and `yh` should have shape `(N,)`.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.3</summary>

```python
yh = theta[0] + theta[1] * x  # b + a*x; one prediction for each input.
plt.plot(x, yh)
plt.scatter(x, y)
plt.xlabel('x')
plt.ylabel('y')
plt.show()
```

</details>


### 1.4) Plotting predictions using matrix-vector multiplication

Now compute the same predictions using the matrix form from the lecture:

$$
\widehat{\mathbf{y}}=\mathbf{X}\boldsymbol{\theta}.
$$

`X` has shape `(N, 2)` and `theta` has shape `(2,)`, so `X @ theta` has shape `(N,)`. Each row of `X` produces one prediction. Plot these predictions with the data. The result should match the previous plot.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.4</summary>

```python
yh = X @ theta         # The same predictions, now computed in matrix form.
plt.plot(x, yh)
plt.scatter(x, y)
plt.xlabel('x')
plt.ylabel('y')
plt.show()
```

</details>


## 2) Fitting a quadratic function

We now use the nonlinear feature construction from the lecture. Consider

$$
f^{\mathrm{quad}}_{\mathbf{w}}(x)
=
w_1+w_2x+w_3x^2.
$$

This predictor is nonlinear in $x$, but it is still **linear in the adjustable coefficients** $w_1,w_2,w_3$. Its feature map is

$$
\boldsymbol{\phi}(x)=(1,\;x,\;x^2).
$$

For the observed inputs, place these feature rows into the transformed design matrix

$$
\boldsymbol{\Phi}
=
\begin{pmatrix}
1 & x_1 & x_1^2\\
1 & x_2 & x_2^2\\
\vdots & \vdots & \vdots\\
1 & x_N & x_N^2
\end{pmatrix}.
$$

Construct this matrix as `Phi`, with shape `(N, 3)`. Then use

$$
\mathbf{w}^*
=
(\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}
\boldsymbol{\Phi}^T\mathbf{y}
$$

to obtain `w`, which has shape `(3,)` in PyTorch. The first coefficient multiplies the constant feature $1$ and therefore plays the role of the intercept.

Compute the predictions as `Phi @ w`, plot them with the observed data, and print `w`.

From this point onward, keep `Phi` and `w` for Section 3.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 2</summary>

```python
ones = t.ones(N)
Phi = t.stack([ones, x, x*x], dim=-1)  # Columns: 1, x, x^2; shape (N, 3).

w = t.linalg.inv(Phi.mT @ Phi) @ (Phi.mT @ y)
yh = Phi @ w

plt.plot(x, yh)
plt.scatter(x, y)
plt.xlabel('x')
plt.ylabel('y')
plt.show()
print(w)
```

</details>


## 3) Computing the optimal coefficients using gradient descent

### 3.1) Computing the loss

Write a function `loss(w)` that takes a coefficient vector and returns the empirical squared loss for the quadratic predictor:

$$
\mathcal{L}^{\phi}_{\mathrm{sq}}(\mathbf{w})
=
\sum_{i=1}^{N} r_i(\mathbf{w})^2,
\qquad
\mathbf{r}(\mathbf{w})
=
\boldsymbol{\Phi}\mathbf{w}-\mathbf{y}.
$$

Thus the residual uses the same sign convention as the lecture: **prediction minus target**.

Use the transformed design matrix `Phi` from Section 2 and the original targets `y`. The input `w` has shape `(3,)`, the residual vector has shape `(N,)`, and the loss should be a scalar tensor.

Use the **sum**, not the mean, of the squared residuals. The gradient formula and learning rate below correspond to this definition.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 3.1</summary>

```python
def loss(w):
    yh = Phi @ w        # Predictions: shape (N,).
    r = yh - y          # Residuals: prediction minus target; shape (N,).
    return (r**2).sum() # Sum the N squared residuals.
```

</details>


### 3.2) Computing the gradient

Write a function `grad(w)` that takes a coefficient vector and returns the gradient of the empirical squared loss at those coefficients. Implement the formula

$$
\nabla \mathcal{L}^{\phi}_{\mathrm{sq}}(\mathbf{w})
=
2\boldsymbol{\Phi}^T
\bigl(\boldsymbol{\Phi}\mathbf{w}-\mathbf{y}\bigr)
=
2\boldsymbol{\Phi}^T\mathbf{r}(\mathbf{w}).
$$

Again, the residual is **prediction minus target**.

The result should have shape `(3,)`: one derivative for each of the three coefficients.

**Self-check:** after you implement `grad`, `grad(t.zeros(3)).shape` should be `torch.Size([3])`.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 3.2</summary>

```python
def grad(w):
    yh = Phi @ w
    r = yh - y
    # Phi.mT has shape (3, N), while r has shape (N,).
    return 2 * (Phi.mT @ r)  # Gradient: shape (3,).
```

</details>


### 3.3) Using gradient descent to find the coefficients

Starting from zero coefficients, apply the update

$$
\mathbf{w}^{(t+1)}
=
\mathbf{w}^{(t)}
-
\eta
\nabla \mathcal{L}^{\phi}_{\mathrm{sq}}
\bigl(\mathbf{w}^{(t)}\bigr),
$$

where $\eta$ is the **learning rate**. Use $\eta=0.001$ and perform 100 updates. Call the iteratively updated coefficients `w_gd`, keeping `w` for the analytic solution from Section 2.

Print the loss after every update to check that it is decreasing. After the 100 updates:

1. print `w_gd` and `w`;
2. compute the loss at `w_gd` and the loss at the analytic solution `w`;
3. print both losses and their difference.

Your two losses should be **very close**, but they need not be identical after only 100 updates. In this dataset they should both be around `7.903`.

**Why does the loss not go to zero?**

If the loss increases instead of decreasing, first check that your update subtracts `eta * grad(w_gd)` rather than adding it.


In [ ]:
# Your code here.
# Suggested structure:
# w_gd = t.zeros(3)
# eta = 0.001
# for _ in range(100):
#     w_gd = w_gd - eta * grad(w_gd)
#     print(loss(w_gd))
#
# analytic_loss = loss(w)
# gd_loss = loss(w_gd)
# print("Gradient descent:", w_gd)
# print("Analytic solution:", w)
# print(f"Loss at the analytic solution: {analytic_loss.item():.6f}")
# print(f"Loss after gradient descent:  {gd_loss.item():.6f}")
# print(f"Difference:                   {(gd_loss - analytic_loss).item():.6f}")


<details>
<summary>Show answer — 3.3</summary>

```python
w_gd = t.zeros(3)
eta = 0.001

# `_` is an ordinary loop variable. We do not use its value here.
for _ in range(100):
    w_gd = w_gd - eta * grad(w_gd)  # Evaluate the gradient at the current coefficients.
    print(loss(w_gd))                # Evaluate the loss at the updated coefficients.

analytic_loss = loss(w)
gd_loss = loss(w_gd)

print("Gradient descent:", w_gd)
print("Analytic solution:", w)
print(f"Loss at the analytic solution: {analytic_loss.item():.6f}")
print(f"Loss after gradient descent:  {gd_loss.item():.6f}")
print(f"Difference:                   {(gd_loss - analytic_loss).item():.6f}")
```

With the supplied dataset, the two losses should be very close, both about `7.903`, but not exactly equal after only 100 updates.

The noisy observations do not all lie on one quadratic curve, so even the analytic optimum has a non-zero loss. More gradient-descent steps can bring the coefficients closer to that optimum. They do not remove this remaining fitting error.

</details>
